# VocEd Lab 05 — Convolutions & a Minimal CNN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/05_convolutions.ipynb)

In Lab 04 we classified pixels by their colour alone.  Each pixel was treated independently
— the classifier had no idea what pixels surrounded it.  In this lab we introduce the
**convolution operation**, which looks at a small *neighbourhood* (a patch) of pixels at
once.  This is the core building block of every modern image recognition system.

We start by applying filters manually so the maths is transparent, then we build a small
**Convolutional Neural Network (CNN)** in PyTorch that learns its own filter weights
directly from the training data.

By the end of this lab you will be able to:
- Compute a convolution by hand using a nested loop.
- Explain the difference between a blur kernel and an edge-detection kernel.
- Build a minimal PyTorch CNN (`nn.Module`) with `Conv2d`, `ReLU`, and a final 1×1 conv.
- Train the CNN on the training split and evaluate on the test split.
- Extend the cumulative Dice table with CNN results.

## 0. Setup

In [ ]:
# Clone the repo
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

## 1. Load Data & Recreate the Train/Test Split

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split

N = len(glob.glob('imagedata/X/*.npy'))
X = np.stack([np.load(f'imagedata/X/{i}.npy') for i in range(N)])
y = np.stack([np.load(f'imagedata/y/{i}.npy') for i in range(N)])

# Drop images with no nucleus pixels — identical to Lab 02
has_nucleus = (y == 2).sum(axis=(1, 2)) > 0
X, y = X[has_nucleus], y[has_nucleus]
N = len(X)

# Stratified split by N/C ratio — identical to Lab 02
nuc_px   = (y == 2).sum(axis=(1, 2))
cyt_px   = (y == 1).sum(axis=(1, 2))
nc_ratio = nuc_px / np.maximum(cyt_px, 1)   # np.maximum avoids div-by-zero
quartile = np.digitize(nc_ratio, np.percentile(nc_ratio, [25, 50, 75]))

train_idx, test_idx = train_test_split(
    np.arange(N), test_size=0.2, stratify=quartile, random_state=42
)

mask_cmap = ListedColormap(['black', 'steelblue', 'crimson'])
legend_patches = [
    mpatches.Patch(color='black',     label='0 — background'),
    mpatches.Patch(color='steelblue', label='1 — cytoplasm'),
    mpatches.Patch(color='crimson',   label='2 — nucleus'),
]
print(f'{N} images retained  ({(~has_nucleus).sum()} with no nucleus removed)')
print(f'Train: {len(train_idx)}   Test: {len(test_idx)}')

## 2. Convolution by Hand

A convolution slides a small **kernel** (a grid of weights) over every position in the
image.  At each position it multiplies the kernel weights with the pixel values underneath
and sums them up.  The result at that position is a single number: the **feature response**.

We work on the grayscale version of image 7 and try three kernels:
1. **Blur kernel** — all positive weights that sum to 1; averages the neighbourhood.
2. **Edge kernel** — detects horizontal edges; positive above, negative below.
3. **Random weights** — what does a kernel with no special structure produce?

Then we ask: *what if the network could learn the best weights by itself?*  That's the
leap to CNNs.

In [ ]:
# ── Manual 3×3 convolution (single-channel, no padding) ──────────────────────
def manual_conv2d(image, kernel):
    """
    Apply a 3×3 kernel to a 2-D image.  Returns an output smaller by 1 pixel
    on each side (no padding).

    image  : (H, W) float array
    kernel : (3, 3) float array
    """
    H, W = image.shape
    output = np.zeros((H - 2, W - 2))   # output shrinks because we need full 3×3 neighbourhoods

    for row in range(H - 2):
        for col in range(W - 2):
            # Extract the 3×3 patch centred at (row+1, col+1)
            patch = image[row : row + 3, col : col + 3]   # shape: (3, 3)
            # Element-wise multiply patch with kernel, then sum — this is a dot product
            output[row, col] = (patch * kernel).sum()

    return output


# Pick one image to demonstrate the kernels on
IDX  = 7
img  = X[IDX]
gray = 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]   # (256, 256)

# ── Three kernels ─────────────────────────────────────────────────────────────
# Blur kernel: uniform average of the 3×3 neighbourhood
blur_kernel = np.ones((3, 3)) / 9.0

# Horizontal edge kernel: highlights rows where intensity changes sharply
# +1 on top, 0 in middle, -1 on bottom → large response at horizontal edges
edge_kernel = np.array([[ 1,  1,  1],
                         [ 0,  0,  0],
                         [-1, -1, -1]], dtype=float)

# Random kernel: random weights, no special structure
rng = np.random.default_rng(7)
rand_kernel = rng.standard_normal((3, 3))

# ── Apply and visualise ───────────────────────────────────────────────────────
# We use a 64×64 crop to keep the manual loop fast
crop = gray[96:160, 96:160]   # (64, 64) crop near the cell

print('Running manual convolutions on a 64×64 crop (may take a few seconds)...')
out_blur = manual_conv2d(crop, blur_kernel)
out_edge = manual_conv2d(crop, edge_kernel)
out_rand = manual_conv2d(crop, rand_kernel)
print('Done.')

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

axes[0, 0].imshow(crop, cmap='gray');     axes[0, 0].set_title('Original crop');  axes[0, 0].axis('off')
axes[0, 1].imshow(out_blur, cmap='gray'); axes[0, 1].set_title('Blur kernel');    axes[0, 1].axis('off')
axes[0, 2].imshow(out_edge, cmap='RdBu_r', vmin=-0.5, vmax=0.5);
axes[0, 2].set_title('Edge kernel');  axes[0, 2].axis('off')
axes[0, 3].imshow(out_rand, cmap='RdBu_r');
axes[0, 3].set_title('Random kernel');  axes[0, 3].axis('off')

# Show the kernel weights themselves
for ax, k, title in zip(axes[1, :3],
                        [blur_kernel, edge_kernel, rand_kernel],
                        ['Blur', 'Edge', 'Random']):
    im = ax.imshow(k, cmap='RdBu_r')
    ax.set_title(f'{title} weights')
    plt.colorbar(im, ax=ax, shrink=0.8)

axes[1, 3].text(0.1, 0.5,
    'The CNN will\nlearn weights like\nthese automatically\nduring training.',
    transform=axes[1, 3].transAxes, fontsize=12, va='center')
axes[1, 3].axis('off')

plt.tight_layout()
plt.show()

## 3. Building a Minimal CNN in PyTorch

The manual loop above is far too slow for training.  PyTorch implements convolutions using
highly optimised GPU/CPU code.  More importantly, PyTorch can **differentiate through**
the convolution — it can compute how much the loss would change if each kernel weight
changed slightly.  This is the gradient we need to train the network.

Our minimal CNN:
```
Input (N, 3, 256, 256)
    → Conv2d(3, 16, 3, padding=1) + ReLU     # 16 filters of size 3×3
    → Conv2d(16, 32, 3, padding=1) + ReLU    # 32 filters of size 3×3
    → Conv2d(32, 3, 1)                        # 1×1 conv: maps 32 channels → 3 class logits
Output (N, 3, 256, 256)  ← one logit per class per pixel
```

`padding=1` keeps the spatial size the same (256×256 in, 256×256 out).  The final 1×1
convolution is just a per-pixel linear layer that mixes the 32 feature channels into 3
class scores.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── Device selection: use GPU if available, otherwise CPU ─────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


# ── Dataset wrapper ───────────────────────────────────────────────────────────
# PyTorch requires a Dataset object that returns (image, label) pairs.
# __len__ returns the number of samples; __getitem__ returns one sample by index.
class SegDataset(Dataset):
    def __init__(self, X, y, indices):
        self.X = X
        self.y = y
        self.indices = indices

    def __len__(self):
        return len(self.indices)   # how many images in this split

    def __getitem__(self, i):
        idx = self.indices[i]
        # torch.from_numpy wraps a NumPy array without copying data
        img  = torch.from_numpy(self.X[idx])   # (3, 256, 256) float32
        mask = torch.from_numpy(self.y[idx].astype(np.int64))  # (256, 256) int64
        return img, mask


# DataLoaders: shuffle training data each epoch; don't shuffle test data
train_loader = DataLoader(SegDataset(X, y, train_idx), batch_size=8, shuffle=True)
test_loader  = DataLoader(SegDataset(X, y, test_idx),  batch_size=8, shuffle=False)

print(f'Training batches: {len(train_loader)}   Test batches: {len(test_loader)}')

In [ ]:
# ── Minimal CNN ───────────────────────────────────────────────────────────────
# nn.Module is the base class for all PyTorch models.
# We define the layers in __init__ and describe the forward pass in forward().
class MinimalCNN(nn.Module):
    def __init__(self):
        super().__init__()   # always call super().__init__() first

        # Conv2d(in_channels, out_channels, kernel_size, padding)
        # padding=1 keeps spatial size the same when kernel_size=3
        self.conv1 = nn.Conv2d(3,  16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        # 1×1 convolution: acts as a per-pixel linear classifier
        # maps 32 feature channels → 3 class logits
        self.conv3 = nn.Conv2d(32,  3, kernel_size=1)

        self.relu = nn.ReLU()   # ReLU(x) = max(0, x) — adds non-linearity

    def forward(self, x):
        # x: (batch, 3, 256, 256)
        x = self.relu(self.conv1(x))   # → (batch, 16, 256, 256)
        x = self.relu(self.conv2(x))   # → (batch, 32, 256, 256)
        x = self.conv3(x)              # → (batch, 3, 256, 256) — logits, no activation
        return x


model = MinimalCNN().to(device)   # .to(device) moves the model to GPU if available
print(model)

# Count total trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

## 4. Training the CNN

In [ ]:
# ── Loss and optimiser ────────────────────────────────────────────────────────
# CrossEntropyLoss expects raw logits (no softmax) and integer labels.
# It is the standard loss for multi-class classification.
criterion = nn.CrossEntropyLoss()

# Adam is a popular gradient-descent optimiser.
# lr=1e-3 is the learning rate — how large each parameter update step is.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


# ── Training loop ─────────────────────────────────────────────────────────────
NUM_EPOCHS = 5   # 5 passes over the training data
train_losses = []

for epoch in range(NUM_EPOCHS):
    model.train()   # puts model in training mode (enables dropout, batch norm tracking, etc.)
    epoch_loss = 0.0

    for imgs, masks in train_loader:
        imgs  = imgs.to(device)    # move batch to GPU/CPU
        masks = masks.to(device)

        # Forward pass: compute predictions
        logits = model(imgs)       # (batch, 3, 256, 256)

        # Compute loss: CrossEntropyLoss compares logits with ground-truth labels
        loss = criterion(logits, masks)

        # Backward pass: compute gradients
        optimizer.zero_grad()   # clear gradients from the previous batch
        loss.backward()         # backpropagate — fills .grad for every parameter

        # Update parameters: move each weight slightly in the direction that reduces loss
        optimizer.step()

        epoch_loss += loss.item()   # .item() converts a 1-element tensor to a Python float

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f'Epoch {epoch+1}/{NUM_EPOCHS}  —  avg loss: {avg_loss:.4f}')

# ── Plot training loss ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, NUM_EPOCHS + 1), train_losses, marker='o', color='steelblue')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('Training Loss Curve — Minimal CNN')
plt.tight_layout()
plt.show()

## 5. Evaluation on the Test Set

In [ ]:
def dice_score(pred, target, cls):
    pred_mask   = (pred   == cls)
    target_mask = (target == cls)
    intersection = (pred_mask & target_mask).sum()
    denom = pred_mask.sum() + target_mask.sum()
    return 1.0 if denom == 0 else 2 * intersection / denom


model.eval()   # puts model in evaluation mode (disables dropout etc.)
cnn_scores = []

# torch.no_grad() tells PyTorch not to build a computation graph —
# we don't need gradients at test time, so this saves memory and speeds up inference.
with torch.no_grad():
    for imgs, masks in test_loader:
        imgs  = imgs.to(device)
        logits = model(imgs)                           # (batch, 3, 256, 256)
        preds  = logits.argmax(dim=1).cpu().numpy()    # take the class with the highest logit
        masks  = masks.numpy()

        for pred, mask in zip(preds, masks):
            d = (dice_score(pred, mask, 1) + dice_score(pred, mask, 2)) / 2
            cnn_scores.append(d)

mean_cnn = np.mean(cnn_scores)
print(f'Minimal CNN test Dice: {mean_cnn:.4f}  ±  {np.std(cnn_scores):.4f}')

In [ ]:
# ── Visual comparison: image 7 + 6 random test images ────────────────────────
def segment_plain(img, t_nucleus=0.45, t_background=0.85):
    gray = 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]
    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[gray < t_nucleus]                                = 2
    pred[(gray >= t_nucleus) & (gray < t_background)]    = 1
    return pred

def segment_lab02(img):
    return segment_plain(img, t_nucleus=0.3904, t_background=0.9900)

# Image 7 on top, then 6 random test images (excluding 7 to avoid duplicates)
rng = np.random.default_rng(0)
candidates = np.array([i for i in test_idx if i != 7])
sample_indices = [7, *rng.choice(candidates, size=6, replace=False).tolist()]

model.eval()
fig, axes = plt.subplots(7, 5, figsize=(20, 28))
for row, idx in enumerate(sample_indices):
    img_t = torch.from_numpy(X[idx]).unsqueeze(0).to(device)
    with torch.no_grad():
        pred_cnn = model(img_t).argmax(dim=1).squeeze().cpu().numpy()
    pred_thresh = segment_plain(X[idx])
    pred_lab02  = segment_lab02(X[idx])

    axes[row, 0].imshow(X[idx].transpose(1, 2, 0));                                                     axes[row, 0].axis('off')
    axes[row, 1].imshow(y[idx],       cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest');         axes[row, 1].axis('off')
    axes[row, 2].imshow(pred_thresh,  cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest');         axes[row, 2].axis('off')
    axes[row, 3].imshow(pred_lab02,   cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest');         axes[row, 3].axis('off')
    axes[row, 4].imshow(pred_cnn,     cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest');         axes[row, 4].axis('off')

    if row == 0:
        axes[0, 0].set_title(f'RGB  (image {idx})')
        axes[0, 1].set_title('Ground truth')
        axes[0, 2].set_title('Lab 01 threshold')
        axes[0, 3].set_title('Lab 02 — Bayesian opt')
        axes[0, 4].set_title('Lab 05 — Minimal CNN')
    else:
        axes[row, 0].set_title(f'image {idx}')

fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, 0.0))
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.show()

# Cumulative Dice table
d_lab01 = np.mean([
    (dice_score(segment_plain(X[i]), y[i], 1) + dice_score(segment_plain(X[i]), y[i], 2)) / 2
    for i in test_idx
])
d_lab02 = np.mean([
    (dice_score(segment_lab02(X[i]), y[i], 1) + dice_score(segment_lab02(X[i]), y[i], 2)) / 2
    for i in test_idx
])
print('\nCumulative Dice comparison (test set):')
print('=' * 50)
print(f'{"Lab 01 — hand-picked thresholds":<35}  {d_lab01:.4f}')
print(f'{"Lab 02 — Bayesian optimisation":<35}  {d_lab02:.4f}')
print(f'{"Lab 05 — Minimal CNN":<35}  {mean_cnn:.4f}')
print('=' * 50)

## 6. N/C Ratio Scatter — Predicted vs True

Dice measures mask overlap.  Here we check whether that improvement in Dice actually
translates into a more accurate **N/C ratio** — the quantity a pathologist would measure.
Points near the y = x line mean the predicted ratio closely matches the ground truth.

In [ ]:
def r2_identity(yp, gt):
    """R² vs y = x: 1 means perfect agreement with the identity line."""
    ss_res = np.sum((yp - gt) ** 2)
    ss_tot = np.sum((gt - gt.mean()) ** 2)
    return 1 - ss_res / ss_tot

def nc_ratio(mask):
    """N/C ratio: nucleus pixel count / cytoplasm pixel count. Returns nan if no cytoplasm."""
    nuc = (mask == 2).sum()
    cyt = (mask == 1).sum()
    return nuc / cyt if cyt > 0 else np.nan


# ── True N/C ratios from ground-truth masks ───────────────────────────────────
nc_true = np.array([nc_ratio(y[i]) for i in test_idx])

# ── Lab 01: hand-picked threshold predictions ─────────────────────────────────
nc_lab01 = np.array([nc_ratio(segment_plain(X[i])) for i in test_idx])

# ── Lab 02: Bayesian-optimised threshold predictions ─────────────────────────
nc_lab02 = np.array([nc_ratio(segment_lab02(X[i])) for i in test_idx])

# ── CNN: run inference one image at a time ────────────────────────────────────
model.eval()
nc_cnn = []
with torch.no_grad():
    for i in test_idx:
        img_t = torch.from_numpy(X[i]).unsqueeze(0).to(device)
        pred  = model(img_t).argmax(dim=1).squeeze().cpu().numpy()
        nc_cnn.append(nc_ratio(pred))
nc_cnn = np.array(nc_cnn)

# ── Scatter plots ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, nc_pred, label, color in zip(
        axes,
        [nc_lab01, nc_lab02, nc_cnn],
        ['Lab 01 — hand-picked', 'Lab 02 — Bayesian opt', 'Lab 05 — Minimal CNN'],
        ['steelblue', 'seagreen', 'crimson']):

    valid = np.isfinite(nc_true) & np.isfinite(nc_pred)
    x, yp = nc_true[valid], nc_pred[valid]

    r2 = r2_identity(yp, x)

    ax.scatter(x, yp, alpha=0.6, color=color, edgecolors='none', s=40)
    lim = max(x.max(), yp.max()) * 1.1
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1.5, label='y = x (perfect)')
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_aspect('equal')
    ax.set_xlabel('True N/C ratio')
    ax.set_ylabel('Predicted N/C ratio')
    ax.set_title(f'{label}\nR² = {r2:.3f}')
    ax.legend(fontsize=9)

plt.suptitle('N/C Ratio: Predicted vs True (test set)', fontsize=13)
plt.tight_layout()
plt.show()

## Wrap-up

**Key takeaways:**
- A convolution looks at a *patch* — it incorporates information from neighbouring pixels,
  which is more powerful than classifying each pixel in isolation.
- The CNN learns its filter weights from the data, so it automatically finds the
  patterns most useful for distinguishing the three classes.
- But our minimal CNN never changes its spatial resolution: every layer sees the same
  256×256 grid.  It has **no ability to zoom out** and see larger structures (e.g., the
  full shape of a cell).  This is the bottleneck we fix in Lab 06 with a U-Net.

---

## Group Exercise — Filter Depth

The number of filters per layer controls how many distinct patterns the CNN can detect.

| Person | Architecture to test |
|---|---|
| A | `Conv2d(3,8,3)` → `Conv2d(8,8,1)` (very shallow) |
| B | `Conv2d(3,16,3)` → `Conv2d(16,32,3)` → `Conv2d(32,3,1)` (this lab's default) |
| C | `Conv2d(3,32,3)` → `Conv2d(32,64,3)` → `Conv2d(64,3,1)` (wider) |

Train each architecture for 5 epochs and record:
- Final training loss.
- Dice on `test_idx[0:5]`.
- Number of trainable parameters.

Discuss: does more depth/width always help?  What is the trade-off?